### Examples using Drain3 to extract URL template from busy URL's
David C. Sisk, 2026/09/30, https://www.linkedin.com/in/davesisk-doctordatabase/

The purpose here is to find and demonstrate an easy way to turn messy URL's and URI's into clusterable data by removing high-cardinality artifacts

In [64]:
#!pip install drain3

In [65]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from pathlib import Path

# Load config from the notebook directory to avoid CWD issues
config = TemplateMinerConfig()
config_path = Path("examples-drain3.ipynb").resolve().parent / "drain3.ini"
config.load(str(config_path))

# Main miner (with masking)
miner = TemplateMiner(persistence_handler=None, config=config)

# Secondary miner with masking disabled for before/after comparison
config_no_mask = TemplateMinerConfig()
config_no_mask.load(str(config_path))
config_no_mask.masking = []
miner_no_mask = TemplateMiner(persistence_handler=None, config=config_no_mask)

print("Drain3 initialized successfully.")

Drain3 initialized successfully.


In [66]:
from urllib.parse import urlsplit, parse_qsl


def normalize_uri(uri):
    """
    Normalize a URI while preserving its structural information.

    Full URLs:
        https://example.com/api/users/123?q=test

    become:
        /api/users/123/user/<*>

    URI paths are left as paths. Query params are converted to masked path segments.
    """

    if uri is None:
        return ""

    uri = str(uri).strip()

    if not uri:
        return ""

    parsed = urlsplit(uri)
    path = parsed.path
    query = parsed.query

    if query:
        params = parse_qsl(query, keep_blank_values=True)
        # convert query params to path-style masked segments: /key/<*>
        suffix = "".join(f"/{k}/<*>" for k, _ in params)
        return f"{path}{suffix}"

    return path

In [67]:
uris = [
    "/api/users/12345/orders/98765",
    "/api/users/67890/orders/12345",
    "/api/users/55555/orders/77777",

    "/api/products/12345",
    "/api/products/67890",
    "/api/products/98765",

    "/api/login",
    "/api/login?user=dave",
    "/api/login?user=john",
    "/api/login?user=mary",

    "/api/files/550e8400-e29b-41d4-a716-446655440000",
    "/api/files/6ba7b810-9dad-11d1-80b4-00c04fd430c8",

    "https://example.com/api/users/12345/orders/99999",
    "https://example.com/api/users/77777/orders/88888",

    "/download/192.168.1.25/file",
    "/download/10.20.30.40/file",

    "/objects/0123456789abcdef0123456789abcdef",
]

In [68]:
records = []

from urllib.parse import urlsplit, parse_qsl

for uri in uris:
    # masked normalized (query values -> /key/<*>)
    normalized_masked = normalize_uri(uri)

    # unmasked normalized (query values -> /key/value)
    parsed = urlsplit(uri)
    path = parsed.path
    query = parsed.query
    if query:
        params = parse_qsl(query, keep_blank_values=True)
        suffix_unmasked = "".join(f"/{k}/{v}" for k, v in params)
        normalized_unmasked = f"{path}{suffix_unmasked}"
    else:
        normalized_unmasked = path

    if not normalized_masked:
        continue

    # With masking (masked input)
    res_mask = miner.add_log_message(normalized_masked)
    template_mask = res_mask["template_mined"]

    # Without masking (unmasked input)
    res_no_mask = miner_no_mask.add_log_message(normalized_unmasked)
    template_no_mask = res_no_mask["template_mined"]

    records.append({
        "original_uri": uri,
        "normalized_masked": normalized_masked,
        "normalized_unmasked": normalized_unmasked,
        "cluster_id": res_mask["cluster_id"],
        "cluster_size": res_mask["cluster_size"],
        "template_masked": template_mask,
        "template_unmasked": template_no_mask,
    })

print(f"Processed {len(records)} URIs")

Processed 17 URIs


In [69]:
import pandas as pd

# Build DataFrame from records
df = pd.DataFrame(records)

# Show before / after templates side-by-side (restore path separators)
df['template_masked_path'] = df['template_masked'].str.replace(' ', '/', regex=False)
df['template_unmasked_path'] = df['template_unmasked'].str.replace(' ', '/', regex=False)

# Use the normalized field name created earlier in the loop (normalized_unmasked)
display(df[["original_uri", "normalized_unmasked", "template_unmasked_path", "template_masked_path", "cluster_id", "cluster_size"]])

,original_uri,normalized_unmasked,template_unmasked_path,template_masked_path,cluster_id,cluster_size
0,/api/users/12345/orders/98765,/api/users/12345/orders/98765,/api/users/<<NUM>>/orders/<<NUM>>,/api/users/<<NUM>>/orders/<<NUM>>,1,1
1,/api/users/67890/orders/12345,/api/users/67890/orders/12345,/api/users/<<NUM>>/orders/<<NUM>>,/api/users/<<NUM>>/orders/<<NUM>>,1,2
2,/api/users/55555/orders/77777,/api/users/55555/orders/77777,/api/users/<<NUM>>/orders/<<NUM>>,/api/users/<<NUM>>/orders/<<NUM>>,1,3
3,/api/products/12345,/api/products/12345,/api/products/<<NUM>>,/api/products/<<NUM>>,2,1
4,/api/products/67890,/api/products/67890,/api/products/<<NUM>>,/api/products/<<NUM>>,2,2
5,/api/products/98765,/api/products/98765,/api/products/<<NUM>>,/api/products/<<NUM>>,2,3
6,/api/login,/api/login,/api/login,/api/login,3,1
7,/api/login?user=dave,/api/login/user/dave,/api/login/user/dave,/api/login/user/<*>,4,1
8,/api/login?user=john,/api/login/user/john,/api/login/user/john,/api/login/user/<*>,4,2
9,/api/login?user=mary,/api/login/user/mary,/api/login/user/mary,/api/login/user/<*>,4,3
